# Alpha Engine E1-A — v5 US87 price-only preflight

**Owner-run, no model/trading execution.** This notebook is specifically for the **v5** `p2_local_market_history_v1` data-only archive. The older `colab_research_exchange_pilot.ipynb` accepts a DIFFERENT `alpha_research_exchange_v1` ZIP format and must **not** be run against v5.

No Colab data copy or run is permitted until a separate, provider-specific, reviewed cloud-compute rights record exists. This is an attestation, not vendor certification. The original frozen v5 dataset remains `local_only` and unchanged. A successful preflight is not PIT/model readiness.

Verified cloud roots: `bundle-manifest.json` SHA256 `d9f4fb2ef7aa2735cfb95436e7eabb744ffb8d6d387c4294626893fc262dab0c`; `us_market_history.zip` SHA256 `93b2d42cebf3ebfac7b80298f6822fdc4305f7550b73ac1b2bb8d272d3fb3de8`. These are the only two v5 files copied to Colab scratch here.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile

# The operator must configure private paths; never commit actual Drive folder IDs,
# source-provider credentials, account data, or legal approval records.
PRIVATE_V5_DIR = Path('CHANGE_ME_PRIVATE_INPUT_PACKAGES/us87-cn130-20261103-local-v5')
PRIVATE_RIGHTS_RECORD = Path('CHANGE_ME_PRIVATE_E1A_US_PRICE_RIGHTS.json')
VERIFIER_COMMIT = 'CHANGE_ME_FULL_40_CHARACTER_PR1368_COMMIT'
EXPECTED_MANIFEST_SHA = 'd9f4fb2ef7aa2735cfb95436e7eabb744ffb8d6d387c4294626893fc262dab0c'
EXPECTED_US_ARCHIVE_SHA = '93b2d42cebf3ebfac7b80298f6822fdc4305f7550b73ac1b2bb8d272d3fb3de8'
OPERATOR_APPROVES_THIS_PRIVATE_CLOUD_SESSION = False

if ('CHANGE_ME' in str(PRIVATE_V5_DIR) or
    'CHANGE_ME' in str(PRIVATE_RIGHTS_RECORD) or
    not __import__('re').fullmatch(r'[a-f0-9]{40}', VERIFIER_COMMIT) or
    not OPERATOR_APPROVES_THIS_PRIVATE_CLOUD_SESSION):
    raise RuntimeError('Operator must configure private rights, pinned verifier commit, and explicitly opt in')

## Review permission BEFORE copying data into a Colab runtime

Connect your private Google Drive in this cell. Review Google's permission screen and inspect the rights record. The record **must** identify each actual US price provider, its terms URL, reason private Colab analysis is authorized, and a timestamp. The record must bind the two immutable SHAs above. Do not manufacture approvals to satisfy this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

r = json.loads(PRIVATE_RIGHTS_RECORD.read_text(encoding='utf-8'))
assert r.get('schema_version') == 'e1a_price_cloud_rights_v1', 'Unknown rights record version'
assert r.get('market') == 'us'
assert r.get('bundle_manifest_sha256') == EXPECTED_MANIFEST_SHA
assert r.get('history_archive_sha256') == EXPECTED_US_ARCHIVE_SHA
assert r.get('cloud_processing_purpose') == 'private_colab_daily_price_research'
assert r.get('owner_approved_private_cloud_processing') is True
assert isinstance(r.get('provider_rights'), list) and r['provider_rights']
assert all(p.get('private_cloud_compute_permitted') is True
           and str(p.get('terms_url','')).startswith('https://')
           and len(str(p.get('review_basis','')).strip()) >= 12
           for p in r['provider_rights'])
print('A separately reviewed owner rights record is present; not vendor-certified')

## One-time Drive → Colab scratch copy, externally pinned SHA

Do not read 200 files through the mounted Drive filesystem. Only the independently pinned market-history archive and root manifest are needed for this narrow preflight.

In [ ]:
scratch = Path('/content/e1a_v5_price_scratch')
scratch.mkdir(parents=True, exist_ok=False)
for name in ('bundle-manifest.json', 'us_market_history.zip'):
    source = PRIVATE_V5_DIR / name
    if not source.is_file():
        raise FileNotFoundError('Expected v5 cloud source missing: ' + name)
    with source.open('rb') as inp, (scratch / name).open('xb') as out:
        shutil.copyfileobj(inp, out, length=1024 * 1024)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as src:
        for part in iter(lambda: src.read(1024 * 1024), b''):
            h.update(part)
    return h.hexdigest()

assert sha256_file(scratch / 'bundle-manifest.json') == EXPECTED_MANIFEST_SHA
assert sha256_file(scratch / 'us_market_history.zip') == EXPECTED_US_ARCHIVE_SHA
print('PASS: independent root manifest and US87 history byte hashes; no study executed')

## Pinned code verification — read v5 in its OWN schema

Use a reviewed full commit SHA containing `src/research/e1a_price_preflight.py`. Do **not** call the older `cloud_exchange.stage`: that assumes `alpha_research_exchange_v1` and attempts to stage data-only projection files into a producer checkout. The v5 archive was produced by an **earlier, separately pinned producer commit**; do not conflate source producer and validator/experiment revision.

In [ ]:
checkout = Path('/content/e1a_validator_checkout')
if checkout.exists():
    raise FileExistsError('Fresh pinned checkout required')
subprocess.run(['git','clone','--quiet','--filter=blob:none','--no-checkout',
                'https://github.com/liuh886/alpha_engine.git',str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',VERIFIER_COMMIT],check=True)
actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip()
assert actual == VERIFIER_COMMIT
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv'],check=True)
subprocess.run(['uv','sync','--frozen'],cwd=checkout,check=True)
cmd = ['uv','run','--frozen','python','-m','src.research.e1a_price_preflight',
       '--manifest',str(scratch/'bundle-manifest.json'),
       '--archive',str(scratch/'us_market_history.zip'),
       '--expected-manifest-sha256',EXPECTED_MANIFEST_SHA,
       '--expected-archive-sha256',EXPECTED_US_ARCHIVE_SHA,
       '--rights-record',str(PRIVATE_RIGHTS_RECORD),
       '--require-cloud-rights']
subprocess.run(cmd,cwd=checkout,check=True)
print('E1-A preflight PASS. Model execution has NOT been authorized or performed.')

## Stop here and hand off

This notebook performs NO strategy/model execution, trade order, Drive return copy, or website update. The local Agent must select a **genuinely runnable existing daily-price study** and reproduce it at one pinned experiment revision locally before the owner independently runs it on Colab. Keep results in `incoming_unreviewed` only after privacy, source, dependency and cost review. See Issue #1369. No auto-change to frozen producer approval flags or US Alpha158 BLOCKED status.